# Dataset 2 — Embedding Model Selection (p = 0)

Same selection logic as Dataset 1's `05_embedding_selection_p0_*`, applied to the **static** Dataset 2.
Compares all embedding candidates — GraphSAGE (feature_based) **v1 + v2** and Node2Vec (network_based) **v1 + v2**,
each at 32 / 64 / 128 dims — and picks the best embedding + model. `ModelTrainer` uses `split='random'`
(same as the combined threshold notebooks). Best model saved to `models/dataset_2/05_a`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import (
    ModelTrainer,
    CLASSICAL_FEATURE_CANDIDATES,
    load_gnn_dataset,
    load_model,
    make_pipeline,
)

pd.set_option('display.max_columns', 200)
TARGET_COL = 'log_systemic_risk_label'
print('Project root:', PROJECT_ROOT)

Project root: /Users/rubenmarques/Documents/Repositórios/Thesis


## Load embedding candidates (v1 + v2, GraphSAGE + Node2Vec, 32/64/128)

In [2]:
DIMS = [32, 64, 128]
EMB_FILES = {}
for v in ['v1', 'v2']:
    for d in DIMS:
        EMB_FILES[f'graphsage_{v}_{d}'] = f'graphsage_{v}_{d}_dataset2_dataset.parquet'
        EMB_FILES[f'node2vec_{v}_{d}']  = f'node2vec_{v}_{d}_dataset2_dataset.parquet'

trainers = {}
for key, fname in EMB_FILES.items():
    edf, ecols = load_gnn_dataset(PROJECT_ROOT, target_col=TARGET_COL, filename=fname)
    trainers[key] = ModelTrainer(df=edf, feature_cols=ecols, target_col=TARGET_COL, split='random')

pd.DataFrame({k: {'n_emb': len(t.feature_cols), 'train': len(t.train_df), 'val': len(t.val_df), 'test': len(t.test_df)}
             for k, t in trainers.items()}).T

,n_emb,train,val,test
graphsage_v1_32,32,1010,217,217
node2vec_v1_32,32,1010,217,217
graphsage_v1_64,64,1010,217,217
node2vec_v1_64,64,1010,217,217
graphsage_v1_128,128,1010,217,217
node2vec_v1_128,128,1010,217,217
graphsage_v2_32,32,1010,217,217
node2vec_v2_32,32,1010,217,217
graphsage_v2_64,64,1010,217,217
node2vec_v2_64,64,1010,217,217


## Define Models

In [3]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]

candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(32,63,16), max_iter=300, activation="relu", learning_rate="adaptive", learning_rate_init=0.001, early_stopping=True, n_iter_no_change=3, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}
list(candidate_models)

['Linear Regression',
 'Ridge',
 'MLP',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost']

## Train all candidates

In [4]:
for key, t in trainers.items():
    t.train_all(candidate_models)
    print(f'\n=== {key} ===')
    display(t.leaderboard().assign(**{'val/train_rmse': lambda d: (d['validation_rmse'] / d['train_rmse']).round(2), 'val/train_mae': lambda d: (d['validation_mae'] / d['train_mae']).round(2)})[DISPLAY_COLS + ['val/train_rmse', 'val/train_mae']])


=== graphsage_v1_32 ===


,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse
0,Gradient Boosting,0.010,0.065,0.030,0.150,5.00
1,Random Forest,0.020,0.063,0.061,0.165,2.70
2,Linear Regression,0.087,0.103,0.161,0.183,1.14
3,MLP,0.122,0.112,0.231,0.198,0.86
4,Ridge,0.085,0.101,0.162,0.199,1.23
5,XGBoost,0.003,0.076,0.008,0.231,28.88



=== node2vec_v1_32 ===


,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse
0,XGBoost,0.004,0.060,0.011,0.168,15.27
1,Random Forest,0.023,0.068,0.069,0.176,2.55
2,Gradient Boosting,0.012,0.087,0.035,0.189,5.40
3,MLP,0.059,0.089,0.125,0.225,1.80
4,Ridge,0.119,0.126,0.190,0.226,1.19
5,Linear Regression,0.119,0.127,0.190,0.228,1.20



=== graphsage_v1_64 ===


...

,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse
0,XGBoost,0.004,0.058,0.009,0.159,17.67
1,Random Forest,0.024,0.067,0.070,0.180,2.57
2,Gradient Boosting,0.009,0.086,0.031,0.184,5.94
3,Ridge,0.121,0.133,0.186,0.243,1.31
4,Linear Regression,0.121,0.135,0.186,0.248,1.33
5,MLP,0.161,0.177,0.235,0.265,1.13



=== graphsage_v2_128 ===


,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse
0,Random Forest,0.018,0.061,0.059,0.171,2.90
1,Gradient Boosting,0.013,0.068,0.044,0.175,3.98
2,Linear Regression,0.057,0.077,0.117,0.175,1.50
3,Ridge,0.052,0.073,0.120,0.187,1.56
4,XGBoost,0.003,0.076,0.008,0.242,30.25
5,MLP,0.250,0.264,0.363,0.385,1.06



=== node2vec_v2_128 ===


,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse
0,Random Forest,0.023,0.065,0.069,0.174,2.52
1,XGBoost,0.003,0.065,0.005,0.186,37.20
2,Gradient Boosting,0.005,0.097,0.022,0.210,9.55
3,MLP,0.060,0.103,0.109,0.259,2.38
4,Ridge,0.105,0.153,0.157,0.371,2.36
5,Linear Regression,0.106,0.158,0.156,0.387,2.48


## Hyperparameter Tuning

In [5]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([np.full(len(trainer.train_df), -1), np.zeros(len(trainer.val_df), dtype=int)])
    search = RandomizedSearchCV(base_model, param_distributions, n_iter=n_iter,
                                cv=PredefinedSplit(split_idx), scoring='neg_root_mean_squared_error',
                                random_state=42, n_jobs=-1)
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    trainer.best_params[name] = search.best_params_
    return search.best_params_

RF_PARAMS = {
    'model__n_estimators': [100, 200, 300, 400, 500, 600],
    'model__max_depth': [None, 5, 10, 15, 20, 30],
    'model__min_samples_leaf': [1, 2, 5, 10, 15, 20],
    'model__min_samples_split': [2, 5, 10, 15, 20],
    'model__max_features': ['sqrt', 'log2', 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    'model__max_iter': [100, 200, 300, 400, 500, 600],
    'model__max_depth': [3, 4, 5, 6, 8, None],
    'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    'model__min_samples_leaf': [5, 10, 20, 50, 100],
    'model__l2_regularization': [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    'model__max_leaf_nodes': [15, 20, 30, 40, 50, 60],
    'model__max_bins': [64, 128, 255],
}
XGB_PARAMS = {
    'model__n_estimators': [100, 200, 400, 600, 800],
    'model__max_depth': [3, 4, 5, 6, 8, 10],
    'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.5, 0.6, 0.7, 0.8, 1.0],
    'model__min_child_weight': [1, 2, 5, 10],
    'model__gamma': [0, 0.1, 0.5, 1.0, 2.0],
    'model__reg_alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    'model__reg_lambda': [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0, 2.0],
}
MLP_PARAMS = {
    'model__hidden_layer_sizes': [(64,), (128,), (256,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    'model__learning_rate_init': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 0.05],
    'model__learning_rate': ['constant', 'adaptive'],
    'model__batch_size': [32, 64, 128, 'auto'],
}

In [6]:
for key, t in trainers.items():
    print(f'Tuning {key} ...')
    tune(t, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  'Random Forest (tuned)')
    tune(t, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  'Gradient Boosting (tuned)')
    tune(t, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, 'XGBoost (tuned)')
    tune(t, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)), MLP_PARAMS, 'MLP (tuned)')

Tuning graphsage_v1_32 ...
Tuning node2vec_v1_32 ...
Tuning graphsage_v1_64 ...
Tuning node2vec_v1_64 ...
Tuning graphsage_v1_128 ...
Tuning node2vec_v1_128 ...
Tuning graphsage_v2_32 ...


/Users/rubenmarques/Documents/Repositórios/Thesis/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Tuning node2vec_v2_32 ...
Tuning graphsage_v2_64 ...
Tuning node2vec_v2_64 ...
Tuning graphsage_v2_128 ...
Tuning node2vec_v2_128 ...


## Compare embeddings & pick best

In [7]:
summary = []
for key, t in trainers.items():
    row = t.leaderboard().iloc[0]
    summary.append({'embedding': key, 'model': row['model'],
                    'train_rmse': row['train_rmse'], 'validation_rmse': row['validation_rmse'],
                    'val/train_rmse': round(row['validation_rmse'] / row['train_rmse'], 2),
                    'train_mae': row['train_mae'], 'validation_mae': row['validation_mae'],
                    'val/train_mae': round(row['validation_mae'] / row['train_mae'], 2)})
summary = pd.DataFrame(summary).sort_values('validation_rmse').reset_index(drop=True)
display(summary)
best = summary.iloc[0]
print('Best embedding+model:', best['embedding'], '|', best['model'], '| val_rmse', best['validation_rmse'])

,embedding,model,train_rmse,validation_rmse,val/train_rmse,validation_mae
0,graphsage_v1_32,XGBoost (tuned),0.073,0.131,1.79,0.057
1,graphsage_v1_64,MLP (tuned),0.137,0.142,1.04,0.063
2,graphsage_v2_128,MLP (tuned),0.135,0.145,1.07,0.063
3,graphsage_v1_128,MLP (tuned),0.123,0.147,1.20,0.071
4,graphsage_v2_64,Linear Regression,0.133,0.147,1.11,0.074
5,graphsage_v2_32,MLP (tuned),0.121,0.148,1.22,0.061
6,node2vec_v2_32,MLP (tuned),0.119,0.157,1.32,0.077
7,node2vec_v2_64,XGBoost,0.009,0.159,17.67,0.058
8,node2vec_v1_128,Random Forest (tuned),0.123,0.161,1.31,0.065
9,node2vec_v2_128,XGBoost (tuned),0.129,0.161,1.25,0.060


Best embedding+model: graphsage_v1_32 | XGBoost (tuned) | val_rmse 0.131


## Save best model

In [8]:
SAVE_DIR = PROJECT_ROOT / 'src' / 'models' / 'dataset_2' / '05_a'
# save the best model for each embedding candidate (per-embedding subfolder)
for key, t in trainers.items():
    t.save_model(t.leaderboard().iloc[0]['model'], SAVE_DIR / key)
print('Overall best:', best['embedding'], '|', best['model'])

Saved 'XGBoost (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_2/05_a/graphsage_v1_32/XGBoost_(tuned).joblib
Saved 'Gradient Boosting (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_2/05_a/node2vec_v1_32/Gradient_Boosting_(tuned).joblib
Saved 'MLP (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_2/05_a/graphsage_v1_64/MLP_(tuned).joblib
Saved 'XGBoost (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_2/05_a/node2vec_v1_64/XGBoost_(tuned).joblib
Saved 'MLP (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_2/05_a/graphsage_v1_128/MLP_(tuned).joblib
Saved 'Random Forest (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_2/05_a/node2vec_v1_128/Random_Forest_(tuned).joblib
Saved 'MLP (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_2/05_a/graphsage_v2_32/MLP_(tuned).joblib